# [0.3] DESI Data: Verify Target Catalogs and Schema

In this notebook, we will review:
- What data do we have available?
- What final catalogs do we want to make?
- What schema should those catalogs have?

## Available data

### Overall catalog taxonomy

To review from *[0.0] Data Exploration*, the available DESI catalogs are structured as follows:

- **6 different surveys**: Commissioning (`cmx`), Survey Validation (`sv1`, `sv2`, `sv3`), Main, and Special.
- Each survey has a few different **programs**  (any of `dark`, `bright`, `backup`, `other`)
- Each program has up to 100 **healpix groups** (defined as `HPIXGROUP = floor(HEALPIX/100)`)
  - **NOTE:** *or so they say--but in `dark/main/` I find there are more groups than this...*
- Each healpix group has **healpix pixels** (these are `nside 64` nested healpix numbers)
- Each healpix directory contains the following files:
  - **coadded spectra** (`coadd-*.fits`)
  - **redshifts**, corresponding to the coadds (`redrock-*.fits`)
  - **spectra**, as individual exposures(`spectra-*fits.hz`)
  - additional data products, such as:
    - additional redshift information (`rrdetails-*.fits`)
    - emission line fits (`emline-*.fits`)
    - per-healpix exposure information (`hpixexp-*.csv`)
    - MgII QSO classifier outputs (`qso_mgii-*.fits`)
    - QuasarNet QSO classifier outputs (`qso_qn-*.fits`)

![desi-catalog-taxonomy-flowchart](images/desi_catalog_taxonomy_light.svg)

The breakdown of targets in the catalogs are as follows:
| | backup | bright | dark | other |
|---|---|---|---|---|
| **cmx** | 0 | 0 | 0 | 5,000 |
| **main** | 1,632,500 | 11,020,470 | 12,778,525 | 0 |
| **special** | 44,905 | 74,412 | 19,500 | 64,428 |
| **sv1** | 110,599 | 239,057 | 371,000 | 143,679 |
| **sv2** | 4,985 | 82,288 | 85,411 | 0 |
| **sv3** | 156,359 | 729,898 | 862,947 | 0 |


### Coadd files

We will examine the coadd file from the **main** survey, **dark** program, healpix group **0**, healpix **0**.

This file is at `/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix/main/dark/0/0/coadd-main-dark-0.fits`, and has been previously discussed in our *[0.0] Data Exploration* notebook.

We will descibe this file with a series of diagrams:

1. **Overview**: the FITS file as a whole, and the five HDU groups (PRIMARY, FIBERMAP, EXP_FIBERMAP, B/R/Z arms, SCORES).
2. **FIBERMAP table** with field names grouped into 7 thematic clusters
3. **EXP_FIBERMAP table + derived columns**: table with fields names grouped into 3 clusters, as well as our planned MJD_MIN / MJD_MAX summary calculation for the output catalog.
4. **SPECTRAL HDUs**: spectral data for for B, R, Z cameras.
5. **SCORES**: SNR and quality metrics.

#### Coadd FITS overview

![coadd_fits_overview](images/coadd_fits_overview.svg)

#### FIBERMAP: Fibermap table

![fibermap_deep_dive](images/fibermap_deep_dive.svg)

#### EXP_FIBERMAP: Exposure fibermap table

![exp_fibermap_derived](images/exp_fibermap_derived.svg)

#### SPECTRAL HDUs: camera spectra for B, R, and Z

![spectral_hdus](images/spectral_hdus.svg)

#### SCORES

![scores_hdu](images/scores_hdu.svg)

### Reshift (redrock) files

We will examine the coadd file from the **main** survey, **dark** program, healpix group **0**, healpix **0**.

This file is at `/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix/main/dark/0/0/redrock-main-dark-0.fits`, and has been previously discussed in our *[0.0] Data Exploration* notebook.

We will descibe this file with a series of diagrams:
1. Overview of FITS file
2. Redshifts table
3. TSNR2 table

#### Redrock FITS overview

![redrock_fits_overview](images/redrock_fits_overview.svg)

#### REDSHIFTS: Redshifts table

![redshifts_deep_dive](images/redshifts_deep_dive.svg)

#### TSNR2: Template signal-to-noise ratio
*(The "2" is because it's SNR squared)*

![tsnr2_deep_dive](images/tsnr2_deep_dive.svg)

## Planned Output

**Current focus for our HATSified DESI catalogs:**
- Coadded spectra
- Accompanying redshift data

**Out of scope (but could be planned for the future):**
- Per-exposure spectra

### Information to take
**Which input catalogs:**
  - The `main` survey
    - and each of its 3 programs (`dark`, `bright`, and `backup`)

**Per-catalog information:**
- Identifiers:
  - `TARGETID` from coadd FITS
  - Maybe: `EXPID` or `NIGHT` from coadd FITS
- Spatial information:
  - `TARGET_RA` from coadd FITS
  - `TARGET_DEC` from coadd FITS
- Temporal information:
  - Maybe: `MJD_MIN`, `MJD_MAX`, the coadd-based MJD range I propose calculating in section Coadd->EXP_FIBERMAP
  - Maybe: some of the coadd provenance information (`COADD_NUMEXP` and `COADD_NUMNIGHT` might be helpful?)
- Coadd spectra: 
  - Each of the wavelength arrays: `B_WAVELENGTH`, `R_WAVELENGTH`, `Z_WAVELENGTH` from coadd FITS
    - Note: I think these will be the same for each row (the camera has a fixed wavelength range). Could we optimize this better?
  - Each of the flux arrays: `B_FLUX`, `R_FLUX`, `Z_FLUX` from coadd FITS
  - Each of the ivar arrays: `B_IVAR`, `R_IVAR`, `Z_IVAR` from coadd FITS
  - Maybe: the bad pixel bitmask arrays?
- Redshifts: 
  - `Z` from redshift FITS
  - `ZERR` from redshift FITS
  - Maybe: more redshift information? the `CHI2` of the fit? the `ZWARN` flag?
- Maybe: SNR score
  - Maybe: the median per-camera SNRs listed in the Coadd->SCORES section?
